# Emoji Classification ? ResNet18 (Colab/Kaggle)Runs full training + inference (no smoke mode). Set `data_dir` to the folder with `train/`, `test/`, `train_labels.csv`.

In [ ]:
import os, random, math, pathlib, jsonimport numpy as np, pandas as pdfrom PIL import Imageimport torchimport torch.nn as nnfrom torch.utils.data import DataLoader, Datasetimport torchvision.transforms as Timport torchvision.models as modelsfrom sklearn.model_selection import train_test_split# ConfigSEED = 42IMG_SIZE = 224BATCH_SIZE = 64VAL_RATIO = 0.2EPOCHS = 8LR = 1e-4WEIGHT_DECAY = 1e-4data_dir = pathlib.Path("/content/data")train_dir = data_dir / "train"test_dir = data_dir / "test"labels_csv = data_dir / "train_labels.csv"# Seedingdef set_seed(seed=SEED):    random.seed(seed); np.random.seed(seed)    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = Falseset_seed()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print("Device:", device)

In [ ]:
# Labels and splitdf = pd.read_csv(labels_csv)label_names = sorted(df["Label"].unique())label_to_idx = {l:i for i,l in enumerate(label_names)}df["label_idx"] = df["Label"].map(label_to_idx)train_df, val_df = train_test_split(df, test_size=VAL_RATIO, stratify=df["Label"], random_state=SEED)print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Classes: {len(label_names)}")

In [ ]:
# Dataset / DataLoadersimagenet_mean = [0.485, 0.456, 0.406]imagenet_std = [0.229, 0.224, 0.225]train_tfms = T.Compose([    T.Resize((IMG_SIZE, IMG_SIZE)),    T.RandomHorizontalFlip(),    T.RandomRotation(10),    T.ToTensor(),    T.Normalize(mean=imagenet_mean, std=imagenet_std),])val_tfms = T.Compose([    T.Resize((IMG_SIZE, IMG_SIZE)),    T.ToTensor(),    T.Normalize(mean=imagenet_mean, std=imagenet_std),])class EmojiDataset(Dataset):    def __init__(self, dataframe, image_dir, label_to_idx, transform):        self.df = dataframe.reset_index(drop=True)        self.image_dir = pathlib.Path(image_dir)        self.label_to_idx = label_to_idx        self.transform = transform    def __len__(self):        return len(self.df)    def __getitem__(self, idx):        row = self.df.iloc[idx]        img_id = str(row["Id"]).zfill(5) + ".png"        label = self.label_to_idx[row["Label"]]        path = self.image_dir / img_id        with Image.open(path) as img:            img = img.convert('RGB')        img = self.transform(img)        return {"image": img, "label": label, "id": path.stem}train_ds = EmojiDataset(train_df, train_dir, label_to_idx, transform=train_tfms)val_ds = EmojiDataset(val_df, train_dir, label_to_idx, transform=val_tfms)train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Model, training, checkpointclass ResNet18Classifier(nn.Module):    def __init__(self, num_classes):        super().__init__()        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)        in_features = self.backbone.fc.in_features        self.backbone.fc = nn.Linear(in_features, num_classes)    def forward(self, x):        return self.backbone(x)model = ResNet18Classifier(num_classes=len(label_names)).to(device)criterion = nn.CrossEntropyLoss()optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)best_val_acc = 0.0ckpt_path = "/content/output/best_resnet18.pth"os.makedirs('/content/output', exist_ok=True)def run_epoch(loader, train=True):    model.train() if train else model.eval()    tot_loss = 0.0; tot_correct = 0; tot = 0    for batch in loader:        imgs = batch["image"].to(device)        labels = batch["label"].to(device)        with torch.set_grad_enabled(train):            logits = model(imgs)            loss = criterion(logits, labels)            if train:                optimizer.zero_grad(); loss.backward(); optimizer.step()        tot_loss += loss.item() * imgs.size(0)        tot_correct += (logits.argmax(1) == labels).sum().item()        tot += imgs.size(0)    return tot_loss/tot, tot_correct/totfor epoch in range(EPOCHS):    train_loss, train_acc = run_epoch(train_loader, train=True)    val_loss, val_acc = run_epoch(val_loader, train=False)    print(f"Epoch {epoch+1}/{EPOCHS} | Train {train_loss:.4f}/{train_acc:.4f} | Val {val_loss:.4f}/{val_acc:.4f}")    if val_acc > best_val_acc:        best_val_acc = val_acc        torch.save(model.state_dict(), ckpt_path)        print(f"Saved new best at {best_val_acc:.4f}")print("Best val acc:", best_val_acc)model.load_state_dict(torch.load(ckpt_path, map_location=device))

In [ ]:
# Inference + submissionmodel.eval()test_paths = sorted(test_dir.glob("*.png"))pred_labels = []ids = []with torch.no_grad():    for i in range(0, len(test_paths), BATCH_SIZE):        batch = test_paths[i:i+BATCH_SIZE]        imgs = []        for p in batch:            with Image.open(p) as img:                img = img.convert('RGB')            imgs.append(val_tfms(img))            ids.append(p.stem)        images = torch.stack(imgs).to(device)        logits = model(images)        pred_idx = logits.argmax(1).cpu().tolist()        pred_labels.extend([label_names[j] for j in pred_idx])submission = pd.DataFrame({"Id": ids, "Label": pred_labels})out_csv = "/content/output/submission.csv"submission.to_csv(out_csv, index=False)print("Saved submission to", out_csv, "rows:", len(submission))